## `2-ANP-2A-RJS` AI 工作流与 LFM 输入路径质控

这个 notebook 只聚焦单井 `2-ANP-2A-RJS`，对当前仓库里并存的两条 LFM 输入链路做质控对比：

- `notebooks/lfm_time_from_wtie@20260413.ipynb`：直接读取 WTIE 导出的 `ai/*.csv`
- `notebooks/lfm_time_from_wtie@20260416.ipynb`：从 LAS 重新构建 `AI(md)`，再做 MD -> TWT 转换，并在 `lfm_time.py` 里继续低通

这里最终只保留三张图，并把曲线名字直接写清楚：

1. `stage1`：MD 域原始波阻抗
2. `stage2` + `stage4`：两条工作流产出的 TWT 域原始波阻抗
3. `stage3` + `stage4_lowpass`：两条工作流低通后的 TWT 域波阻抗


## 代码路径梳理

根据当前仓库代码，已确认的处理顺序如下：

- `notebooks/auto_well_tie@20260403.ipynb` 会导出 `outputs.logset_twt.AI`
- `wtie.optimize.autotie._common_steps_tie_v1` 的顺序是 `filter_md_logs -> convert_logs_from_md_to_twt -> compute_reflectivity`
- `wtie.optimize.tie.convert_logs_from_md_to_twt` 会继续调用 `wtie.optimize.logs.convert_logs_from_md_to_twt`，并把 `wellpath` 传进去
- `wtie.processing.grid.convert_log_from_md_to_twt` 在 `trajectory is not None` 时会切到轨迹辅助转换路径
- `notebooks/lfm_time_from_wtie@20260413.ipynb` 直接把 WTIE 导出的 `ai/*.csv` 读成 TWT 域 `AI`
- `notebooks/lfm_time_from_wtie@20260416.ipynb` 会重新从 LAS 派生 `AI(md)`，再交给 `build_lfm_time_model`
- `cup.seismic.lfm_time._prepare_well` 的顺序是先 `convert_log_from_md_to_twt`，再 `lowpass_twt_log`

这里要特别注意：

- 另一个工作流产出的 TWT 域波阻抗，是很有价值的旧链路对照曲线，但它**不是**当前 `20260416` 分支的原样拷贝。
- WTIE 导出链路里还包含 MD 域滤波与轨迹辅助转换，而当前 `20260416` 分支使用的是 MD 域 checkshot 直接转换。


In [ ]:
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from matplotlib import font_manager

repo_root = Path.cwd()
if not (repo_root / "src").exists():
    repo_root = repo_root.parent
if not (repo_root / "src").exists():
    raise RuntimeError("Could not locate repo root containing 'src'.")

src_root = repo_root / "src"
if str(src_root) not in sys.path:
    sys.path.insert(0, str(src_root))

from cup.petrel.load import import_checkshots_petrel, load_vp_rho_logset_from_las
from cup.seismic.lfm_time import lowpass_twt_log
from wtie.processing import grid

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.float_format", lambda x: f"{x:,.6f}")


WELL_NAME = "L1-NW1"
INTERPRETATION_OFFSET = 0.03
DT_TWT = 0.002
LOWPASS_CUTOFF_HZ = 10.0
LOWPASS_ORDER = 5

CURVE_LABELS = {
    "md_raw": "MD 域原始波阻抗",
    "current_twt": "TWT 域原始波阻抗（直接时深转换）",
    "current_twt_lowpass": "直接时深转换 + 低通",
    "other_twt": "TWT 域原始波阻抗（自动井震标定产出）",
    "other_twt_lowpass": "自动井震标定产出 + 低通",
}

las_file = repo_root / "data" / "vertical_well_las_target_clear" / f"{WELL_NAME}.las"
tdtable_file = repo_root / "data" / "output_vertical_well_auto_tie_20260417" / "tdtable" / f"{WELL_NAME}_{INTERPRETATION_OFFSET}.txt"
ai_csv_file = repo_root / "data" / "output_vertical_well_auto_tie_20260417" / "ai" / f"{WELL_NAME}_{INTERPRETATION_OFFSET}.csv"

for path in [las_file, tdtable_file, ai_csv_file]:
    if not path.exists():
        raise FileNotFoundError(path)


def configure_matplotlib_fonts() -> str | None:
    candidate_fonts = [
        "Microsoft YaHei",
        "SimHei",
        "Noto Sans CJK SC",
        "Source Han Sans SC",
        "PingFang SC",
        "WenQuanYi Zen Hei",
        "Arial Unicode MS",
    ]
    available_fonts = {font.name for font in font_manager.fontManager.ttflist}
    for font_name in candidate_fonts:
        if font_name in available_fonts:
            plt.rcParams["font.family"] = "sans-serif"
            plt.rcParams["font.sans-serif"] = [font_name, *plt.rcParams.get("font.sans-serif", [])]
            plt.rcParams["axes.unicode_minus"] = False
            return font_name
    plt.rcParams["axes.unicode_minus"] = False
    return None


SELECTED_CJK_FONT = configure_matplotlib_fonts()


def load_ai_log_from_csv(csv_file: Path) -> grid.Log:
    df = pd.read_csv(csv_file)
    required_cols = {"twt_s", "ai"}
    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(f"AI CSV is missing columns {sorted(missing)}: {csv_file}")

    twt = df["twt_s"].to_numpy(dtype=float)
    ai = df["ai"].to_numpy(dtype=float)
    order = np.argsort(twt)
    twt = twt[order]
    ai = ai[order]
    return grid.Log(ai, twt, "twt", name="AI", allow_nan=False)


def safe_sampling_rate(log: grid.Log) -> float:
    if len(log.basis) <= 1:
        return float("nan")
    return float(log.sampling_rate)


def summarize_log(log: grid.Log, curve_name: str, domain: str, source: str, processing: str) -> dict:
    values = np.asarray(log.values, dtype=float)
    basis = np.asarray(log.basis, dtype=float)
    return {
        "curve": curve_name,
        "domain": domain,
        "source": source,
        "processing": processing,
        "n_samples": int(len(log)),
        "sampling": safe_sampling_rate(log),
        "basis_start": float(basis[0]),
        "basis_end": float(basis[-1]),
        "value_min": float(np.nanmin(values)),
        "value_max": float(np.nanmax(values)),
        "value_mean": float(np.nanmean(values)),
    }


def convert_md_ai_to_twt_with_checkshot(
    ai_md: grid.Log, td_table_md: grid.TimeDepthTable, dt: float
) -> tuple[grid.Log, dict]:
    dz = float(ai_md.sampling_rate)
    table_at_dz = td_table_md.depth_interpolation(dz)
    log_md = np.asarray(ai_md.basis, dtype=float)
    table_md = np.asarray(table_at_dz.depth, dtype=float)
    overlap_min = max(float(log_md[0]), float(table_md[0]))
    overlap_max = min(float(log_md[-1]), float(table_md[-1]))
    if overlap_max < overlap_min:
        raise ValueError("Log MD range and time-depth table MD range do not overlap.")

    overlap_mask = (table_md >= overlap_min) & (table_md <= overlap_max)
    if np.count_nonzero(overlap_mask) < 2:
        raise ValueError("Insufficient overlapping MD samples to convert log from MD to TWT.")

    overlap_md = table_md[overlap_mask]
    overlap_twt = np.asarray(table_at_dz.twt, dtype=float)[overlap_mask]
    overlap_values = np.interp(overlap_md, log_md, np.asarray(ai_md.values, dtype=float))
    regular_twt = np.arange(overlap_twt[0], overlap_twt[-1] + dt, dt)
    interp = grid._interp1d(
        overlap_twt,
        overlap_values,
        bounds_error=False,
        fill_value=(overlap_values[0], overlap_values[-1]),  # type: ignore
        kind="linear",
    )
    reproduced = grid.Log(
        interp(regular_twt),
        regular_twt,
        "twt",
        name=ai_md.name,
        unit=getattr(ai_md, "unit", None),
        allow_nan=ai_md.allow_nan,
    )

    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        direct = grid.convert_log_from_md_to_twt(ai_md, td_table_md, None, dt=dt)  # type: ignore

    if len(direct) != len(reproduced):
        reproduction_max_abs_diff = float("nan")
    else:
        reproduction_max_abs_diff = float(np.max(np.abs(np.asarray(direct.values) - np.asarray(reproduced.values))))

    diagnostics = {
        "ai_md_start": float(ai_md.basis[0]),
        "ai_md_end": float(ai_md.basis[-1]),
        "td_md_start": float(td_table_md.depth[0]),
        "td_md_end": float(td_table_md.depth[-1]),
        "matched_md_at_idx_start": float(overlap_md[0]),
        "start_md_delta": float(overlap_md[0] - ai_md.basis[0]),
        "truncation_occurred": bool(overlap_max < float(log_md[-1])),
        "truncated_md_end": float(overlap_md[-1]),
        "converted_twt_start": float(direct.basis[0]),
        "converted_twt_end": float(direct.basis[-1]),
        "converted_twt_samples": int(len(direct)),
        "converted_twt_sampling": float(direct.sampling_rate),
        "warning_messages": [str(item.message) for item in caught],
        "reproduction_max_abs_diff": reproduction_max_abs_diff,
    }
    return direct, diagnostics


def compare_logs_on_common_twt(
    log_a: grid.Log,
    log_b: grid.Log,
    comparison_name: str,
    label_a: str = "log_a",
    label_b: str = "log_b",
) -> tuple[pd.DataFrame, dict]:
    start = max(float(log_a.basis[0]), float(log_b.basis[0]))
    end = min(float(log_a.basis[-1]), float(log_b.basis[-1]))
    dt = min(float(log_a.sampling_rate), float(log_b.sampling_rate))
    if end < start:
        raise ValueError(f"No overlapping TWT support between {label_a} and {label_b}.")

    common_twt = np.arange(start, end + 0.5 * dt, dt)
    values_a = np.interp(common_twt, np.asarray(log_a.basis, dtype=float), np.asarray(log_a.values, dtype=float))
    values_b = np.interp(common_twt, np.asarray(log_b.basis, dtype=float), np.asarray(log_b.values, dtype=float))
    diff = values_a - values_b
    max_idx = int(np.argmax(np.abs(diff)))

    diff_df = pd.DataFrame(
        {
            "twt_s": common_twt,
            label_a: values_a,
            label_b: values_b,
            "diff": diff,
        }
    )
    stats = {
        "pair": comparison_name,
        "n_common_samples": int(len(common_twt)),
        "common_start_twt_s": float(common_twt[0]),
        "common_end_twt_s": float(common_twt[-1]),
        "common_dt_s": float(dt),
        "max_abs_diff": float(np.max(np.abs(diff))),
        "mean_abs_diff": float(np.mean(np.abs(diff))),
        "rmse": float(np.sqrt(np.mean(diff**2))),
        "mean_signed_diff": float(np.mean(diff)),
        "max_abs_diff_twt_s": float(common_twt[max_idx]),
    }
    return diff_df, stats


def build_summary_markdown(
    conversion_diag: dict,
    comp_raw_twt: dict,
    comp_lowpass_twt: dict,
    comp_current_lowpass: dict,
    comp_other_lowpass: dict,
    selected_font: str | None,
) -> str:
    start_diff_ms = 1000.0 * (conversion_diag["other_workflow_twt_start"] - conversion_diag["converted_twt_start"])
    lines = ["## 精简结论", ""]

    if conversion_diag["truncation_occurred"]:
        lines.append("- 当前工作流在 MD -> TWT 转换时发生了有效区间截断，需要继续重点关注这一步。")
    else:
        lines.append("- 当前工作流在 MD -> TWT 转换时没有发生截断。")

    lines.append(
        f"- 当前工作流与另一个工作流的原始 TWT 波阻抗 RMSE 为 {comp_raw_twt['rmse']:.3f}；两边都低通后 RMSE 为 {comp_lowpass_twt['rmse']:.3f}。"
    )
    lines.append(
        f"- 当前工作流内部低通前后 RMSE 为 {comp_current_lowpass['rmse']:.3f}；另一个工作流内部低通前后 RMSE 为 {comp_other_lowpass['rmse']:.3f}。"
    )
    lines.append(f"- 两条原始 TWT 曲线的起始时间相差 {start_diff_ms:.3f} ms。")

    warning_count = len(conversion_diag["warning_messages"])
    if warning_count:
        lines.append(
            f"- MD -> TWT 转换共捕获到 {warning_count} 条 warning；需要时可以直接查看 `conversion_diag['warning_messages']`。"
        )

    if selected_font:
        lines.append(f"- 已显式指定 Matplotlib 中文字体：`{selected_font}`，用于修复图中文字显示问题。")
    else:
        lines.append("- 当前环境未找到常见中文字体，图中文字仍可能依赖默认字体。")

    return "\n".join(lines)


def plot_impedance(ax, log: grid.Log, label: str, color: str, linewidth: float = 1.8, alpha: float = 1.0) -> None:
    ax.plot(log.values, log.basis, label=label, color=color, linewidth=linewidth, alpha=alpha)


In [ ]:
md_ai_log = load_vp_rho_logset_from_las(las_file).AI
td_table_md = import_checkshots_petrel(tdtable_file, depth_domain="md")
current_twt_ai, conversion_diag = convert_md_ai_to_twt_with_checkshot(md_ai_log, td_table_md, dt=DT_TWT)
current_twt_ai_lowpass = lowpass_twt_log(current_twt_ai, cutoff_hz=LOWPASS_CUTOFF_HZ, order=LOWPASS_ORDER)
other_twt_ai = load_ai_log_from_csv(ai_csv_file)
other_twt_ai_lowpass = lowpass_twt_log(other_twt_ai, cutoff_hz=LOWPASS_CUTOFF_HZ, order=LOWPASS_ORDER)

conversion_diag["other_workflow_twt_start"] = float(other_twt_ai.basis[0])
conversion_diag["other_workflow_twt_end"] = float(other_twt_ai.basis[-1])
conversion_diag["other_workflow_twt_sampling"] = float(other_twt_ai.sampling_rate)

curve_overview_df = pd.DataFrame(
    [
        summarize_log(md_ai_log, CURVE_LABELS["md_raw"], "MD", "LAS: Vp * Rho", "原始曲线"),
        summarize_log(current_twt_ai, CURVE_LABELS["current_twt"], "TWT", "当前工作流", "MD -> TWT"),
        summarize_log(
            current_twt_ai_lowpass,
            CURVE_LABELS["current_twt_lowpass"],
            "TWT",
            "当前工作流",
            f"lowpass({LOWPASS_CUTOFF_HZ:.0f} Hz, order={LOWPASS_ORDER})",
        ),
        summarize_log(other_twt_ai, CURVE_LABELS["other_twt"], "TWT", "另一个工作流", "直接读取 ai/*.csv"),
        summarize_log(
            other_twt_ai_lowpass,
            CURVE_LABELS["other_twt_lowpass"],
            "TWT",
            "另一个工作流",
            f"lowpass({LOWPASS_CUTOFF_HZ:.0f} Hz, order={LOWPASS_ORDER})",
        ),
    ]
)

_, comp_raw_twt = compare_logs_on_common_twt(
    current_twt_ai,
    other_twt_ai,
    comparison_name="原始 TWT：当前工作流 vs 另一个工作流",
    label_a="current_twt",
    label_b="other_twt",
)
_, comp_lowpass_twt = compare_logs_on_common_twt(
    current_twt_ai_lowpass,
    other_twt_ai_lowpass,
    comparison_name="低通后 TWT：当前工作流 vs 另一个工作流",
    label_a="current_twt_lowpass",
    label_b="other_twt_lowpass",
)
_, comp_current_lowpass = compare_logs_on_common_twt(
    current_twt_ai,
    current_twt_ai_lowpass,
    comparison_name="当前工作流：低通前 vs 低通后",
    label_a="current_twt",
    label_b="current_twt_lowpass",
)
_, comp_other_lowpass = compare_logs_on_common_twt(
    other_twt_ai,
    other_twt_ai_lowpass,
    comparison_name="另一个工作流：低通前 vs 低通后",
    label_a="other_twt",
    label_b="other_twt_lowpass",
)

comparison_df = pd.DataFrame([comp_raw_twt, comp_lowpass_twt, comp_current_lowpass, comp_other_lowpass])[
    ["pair", "n_common_samples", "rmse", "max_abs_diff", "mean_abs_diff", "mean_signed_diff"]
]

display(Markdown("## 精简质控摘要"))
display(
    curve_overview_df[["curve", "domain", "source", "processing", "n_samples", "sampling", "basis_start", "basis_end"]]
)
display(comparison_df)
display(
    Markdown(
        build_summary_markdown(
            conversion_diag,
            comp_raw_twt,
            comp_lowpass_twt,
            comp_current_lowpass,
            comp_other_lowpass,
            SELECTED_CJK_FONT,
        )
    )
)


In [ ]:
fig, ax = plt.subplots(figsize=(7, 9), constrained_layout=True)

plot_impedance(ax, md_ai_log, CURVE_LABELS["md_raw"], color="#1f77b4", linewidth=1.4)
ax.invert_yaxis()
ax.set_title("MD 域原始波阻抗")
ax.set_xlabel("波阻抗 AI")
ax.set_ylabel("MD (m)")

plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(8, 9), constrained_layout=True)

plot_impedance(ax, current_twt_ai, CURVE_LABELS["current_twt"], color="#d62728", linewidth=1.8)
plot_impedance(ax, other_twt_ai, CURVE_LABELS["other_twt"], color="#9467bd", linewidth=1.6, alpha=0.9)
ax.invert_yaxis()
ax.set_title("TWT 域原始波阻抗")
ax.set_xlabel("波阻抗 AI")
ax.set_ylabel("TWT (s)")
ax.legend(loc="best")

plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(8, 9), constrained_layout=True)

plot_impedance(ax, current_twt_ai_lowpass, CURVE_LABELS["current_twt_lowpass"], color="#2ca02c", linewidth=1.8)
plot_impedance(ax, other_twt_ai_lowpass, CURVE_LABELS["other_twt_lowpass"], color="#ff7f0e", linewidth=1.6, alpha=0.95)
ax.invert_yaxis()
ax.set_title("TWT 域波阻抗 (低通后)")
ax.set_xlabel("波阻抗 AI")
ax.set_ylabel("TWT (s)")
ax.legend(loc="best")

plt.show()
